In [ ]:
# %%
import sys; sys.path.insert(0, '..')


<!-- # %% [markdown] -->
## 1. The oceanography analogy

Imagine an ocean with a hidden subsurface vortex. At the surface, local flow can look calm, so pointwise measurements may miss the structure entirely. Oceanographers often detect such hidden structure by integrating circulation along a closed path: if there is no singular rotational source inside the loop, the net circulation cancels; if a vortex is enclosed, the loop integral is non-zero.

In complex analysis, the same idea is formalized by the Residue Theorem:

$$\oint_C f(z)\,dz = 2\pi i \sum_k \operatorname{Res}(f, z_k).$$

The key conceptual bridge for `cfad` is this: if $f$ is entire (analytic everywhere), the sum over enclosed singularities is empty, so the contour integral is exactly zero. A non-zero contour integral therefore indicates that the enclosed region contains non-analytic structure.


<!-- # %% [markdown] -->
## 2. The characteristic function

We compare three characteristic-function (CF) families on the same frequency grid.

- Gaussian $N(0, 0.01^2)$: entire CF (`is_analytic=True`)
- NIG with $(\alpha,\beta,\delta,\mu)=(10,0,0.1,0)$: branch-cut CF (`is_analytic=False`)
- L'evy-stable with $(\alpha,\beta,c,\mu)=(1.7,0,0.01,0)$: branch-cut CF (`is_analytic=False`)

We plot $\Re[\varphi(\xi)]$ side by side to inspect their frequency-domain geometry.


In [ ]:
# %%
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import norminvgauss, levy_stable

from cfad.models.gaussian import GaussianCF
from cfad.models.nig import NIGCF
from cfad.models.levy_stable import LevyStableCF
from cfad.empirical_cf import ecf_at
from cfad.contour import ecf_residue_scores

xi = np.linspace(-10.0, 10.0, 500)

models = [
    ("Gaussian N(0, 0.01^2)", GaussianCF(mu=0.0, sigma=0.01)),
    ("NIG(alpha=10, beta=0, delta=0.1, mu=0)", NIGCF(alpha=10.0, beta=0.0, delta=0.1, mu=0.0)),
    ("LevyStable(alpha=1.7, beta=0, c=0.01)", LevyStableCF(alpha=1.7, beta=0.0, c=0.01, mu=0.0)),
]

fig_cf, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, (label, model) in zip(axes, models):
    phi = model.cf(xi)
    ax.plot(xi, np.real(phi), color='tab:blue', linewidth=1.5)
    ax.axhline(0.0, color='black', linewidth=0.8, alpha=0.5)
    ax.set_title(f"{label}\nis_analytic={model.is_analytic}")
    ax.set_xlabel(r"$\xi$")
    ax.grid(alpha=0.25)
axes[0].set_ylabel(r"$\Re[\varphi(\xi)]$")
fig_cf.suptitle("Real Part of Characteristic Functions", y=1.03)
fig_cf.tight_layout()
fig_cf


<!-- # %% [markdown] -->
## 3. The contour integral as a detector

Now we simulate returns with $n=300$ observations per replicate and compute one contour score per series.

For each replicate:
- estimate the empirical characteristic function with `ecf_at`
- convert it into a contour-residue score with `ecf_residue_scores`

We repeat this 50 times for each distribution and visualize mean score $\pm$ one standard deviation.


In [ ]:
# %%
rng = np.random.default_rng(2026)
n = 300
n_rep = 50
contour_height = 0.1
xi_score = np.linspace(-10.0, 10.0, 2001)

def contour_score(sample: np.ndarray) -> float:
    phi_hat = ecf_at(sample.astype(np.float64), xi_score)
    score = ecf_residue_scores(phi_hat[np.newaxis, :], xi_score, height=contour_height)[0]
    return float(score)

gaussian_scores = []
nig_scores = []
levy_scores = []

for _ in range(n_rep):
    g = rng.normal(0.0, 0.01, n)
    nig = norminvgauss.rvs(a=10.0, b=0.0, loc=0.0, scale=0.1, size=n, random_state=rng)
    lev = levy_stable.rvs(alpha=1.7, beta=0.0, loc=0.0, scale=0.01, size=n, random_state=rng)

    gaussian_scores.append(contour_score(g))
    nig_scores.append(contour_score(nig))
    levy_scores.append(contour_score(lev))

score_groups = {
    'Gaussian': np.asarray(gaussian_scores, dtype=np.float64),
    'NIG': np.asarray(nig_scores, dtype=np.float64),
    'Levy-stable': np.asarray(levy_scores, dtype=np.float64),
}

for name, values in score_groups.items():
    print(f"{name:12s} mean={values.mean():.6f}, std={values.std():.6f}")

labels = list(score_groups.keys())
means = [score_groups[k].mean() for k in labels]
stds = [score_groups[k].std() for k in labels]

fig_scores, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, means, yerr=stds, capsize=6, color=['tab:green', 'tab:orange', 'tab:red'])
ax.set_ylabel('Contour Residue Score')
ax.set_title('Mean ± Std of Contour Scores Across 50 Replicates')
ax.grid(axis='y', alpha=0.3)

y_offset = 0.03 * max(means) if max(means) > 0 else 0.01
for bar, mean_val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, mean_val + y_offset, f"{mean_val:.3f}",
            ha='center', va='bottom', fontsize=9)

fig_scores.tight_layout()
fig_scores


<!-- # %% [markdown] -->
## 4. The structural gag order

A central message of this notebook is methodological, not cosmetic. If your modeling family is constrained to be analytic in the relevant complex region, then certain structural breaks cannot appear in your detector, no matter how much data you collect.

> "Choosing an analytic CF family is not a simplification — it is a structural constraint that makes certain breaks impossible to detect by construction."

In that sense, model class choice acts like a gag order: it can suppress the very singular behavior you are trying to monitor. `cfad` addresses this by explicitly tracking contour-based evidence of non-analytic structure in rolling windows.


<!-- # %% [markdown] -->
## 5. Save figure

Save the bar chart to `paper/figures/concept_scores.png` at 150 dpi.


In [ ]:
# %%
from pathlib import Path

out_path = Path('../paper/figures/concept_scores.png')
out_path.parent.mkdir(parents=True, exist_ok=True)
fig_scores.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved: {out_path.resolve()}')
